# KSL 수어 인식 - MediaPipe DTW+KNN
gabguerin/Sign-Language-Recognition--MediaPipe-DTW 기반

**데이터 형식**: `.npy` 파일, shape = `(frames, 67, 3)`
- `0:21` → 왼손 21개 키포인트
- `21:42` → 오른손 21개 키포인트
- `42:67` → 포즈 25개 키포인트

**폴더 구조**:
```
KSL_data/
  가/  → 가_001.npy, 가_002.npy ...
  나/  → 나_001.npy ...
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 셀 1: 패키지 설치
!pip install dtaidistance scikit-learn joblib -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 38.6 MB/s eta 0:00:00


In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os

# 내 드라이브 전체 최상위 폴더 목록 확인
print(os.listdir('/content/drive/MyDrive'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['동영상', '인공지능', 'Untitled0.ipynb', 'Untitled', 'Composition_of_functions_vs_Decorator_patterns.ipynb', '이윤승-2143945.ipynb', '2143945-3-1', 'Colab Notebooks', '25-26 GDGoC DAU member introduction의 사본 (3).gslides', '25-26 GDGoC DAU member introduction의 사본 (2).gslides', '25-26 GDGoC DAU member introduction의 사본 (1).gslides', '25-26 GDGoC DAU member introduction의 사본.gslides', '경품 수령 확인 및 기타소득에 대한 개인 정보 수집 및 이용 동의서_이윤승.docx', '최종이었으면 좋겠다.ipynb', 'Untitled1.ipynb', 'global_challengers_2차발표 (2).pptx', 'global_challengers_2차발표 (1).pptx', '글챌 테스트.mp4', 'global_challengers_2차발표.pptx', '2026_1학기_사물인터넷_중간고사.gdoc', '컴퓨터비전_중간고사.gdoc', 'TNF알파_사이토카인보고서_AI학과_2143945_이윤승.gdoc', 'KSL_글래스_프로젝트_타당성보고서 (4)

In [ ]:
# 셀 2: Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

# ↓ 팀원 공유 폴더 경로로 수정하세요
DATA_ROOT = '/content/drive/MyDrive/수어영상'

In [ ]:
# 셀 3: 임포트 및 함수 정의
import os
import numpy as np
import joblib
from sklearn.model_selection import LeaveOneOut
from dtaidistance import dtw

# ── 관절 각도 특징 추출 (위치/크기 불변) ──
HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20)
]

def angle_between(v1, v2):
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 < 1e-6 or n2 < 1e-6:
        return 0.0
    return np.arccos(np.clip(np.dot(v1, v2) / (n1 * n2), -1.0, 1.0))

def extract_hand_angles(hand_kp):
    angles = []
    for (parent, child) in HAND_CONNECTIONS:
        if parent == 0:
            angles.append(np.linalg.norm(hand_kp[child] - hand_kp[parent]))
        else:
            v1 = hand_kp[parent] - hand_kp[parent - 1]
            v2 = hand_kp[child] - hand_kp[parent]
            angles.append(angle_between(v1, v2))
    return np.array(angles, dtype=np.float32)

def extract_features(sequence):
    """sequence: (frames, 67, 3) → (frames, 40)"""
    return np.array([
        np.concatenate([
            extract_hand_angles(sequence[f, 0:21, :]),
            extract_hand_angles(sequence[f, 21:42, :])
        ])
        for f in range(sequence.shape[0])
    ], dtype=np.float32)

# ── 데이터 증강 ──
def augment_sequence(sequence, n=15):
    augmented = []
    for _ in range(n):
        aug = sequence.copy()
        aug += np.random.normal(0, 0.005, aug.shape).astype(np.float32)
        scale = np.random.uniform(0.8, 1.2)
        new_len = max(5, int(aug.shape[0] * scale))
        aug = aug[np.linspace(0, aug.shape[0]-1, new_len).astype(int)]
        if np.random.random() < 0.5:
            tmp = aug.copy()
            tmp[:, 0:21, :] = aug[:, 21:42, :]
            tmp[:, 21:42, :] = aug[:, 0:21, :]
            tmp[:, :42, 0] = 1.0 - tmp[:, :42, 0]
            aug = tmp
        augmented.append(aug)
    return augmented

def augment_dataset(X, y, n=15):
    X_aug, y_aug = list(X), list(y)
    for seq, label in zip(X, y):
        X_aug.extend(augment_sequence(seq, n))
        y_aug.extend([label] * n)
    print(f'증강: {len(X)}개 → {len(X_aug)}개')
    return X_aug, y_aug

# ── DTW + KNN 분류기 ──
class DTWKNNClassifier:
    def __init__(self, k=5):
        self.k = k
        self.templates = []

    def fit(self, X_feat, y):
        self.templates = list(zip(X_feat, y))

    def _dist(self, s1, s2):
        return np.mean([
            dtw.distance_fast(s1[:, d].astype(np.double), s2[:, d].astype(np.double))
            for d in range(s1.shape[1])
        ])

    def predict_one(self, query):
        dists = sorted([(self._dist(query, t), l) for t, l in self.templates])
        votes = {}
        for d, l in dists[:self.k]:
            votes[l] = votes.get(l, 0) + 1.0 / (d + 1e-6)
        best = max(votes, key=votes.get)
        conf = votes[best] / sum(votes.values())
        return best, conf, dists[:5]

# ── 데이터 로드 ──
def load_dataset(root):
    X, y = [], []
    labels = sorted([l for l in os.listdir(root)
                     if os.path.isdir(os.path.join(root, l))])
    for label in labels:
        for f in os.listdir(os.path.join(root, label)):
            if not f.endswith('.npy'):
                continue
            seq = np.load(os.path.join(root, label, f)).astype(np.float32)
            if seq.ndim == 2:
                seq = seq.reshape(seq.shape[0], 67, 3)
            X.append(seq)
            y.append(label)
    print(f'로드: {len(X)}개 샘플 / {len(labels)}개 클래스')
    print(f'클래스: {labels}')
    return X, y, labels

print('함수 정의 완료!')

In [ ]:
# 셀 4: 데이터 로드 + 특징 추출
X_raw, y_raw, label_names = load_dataset(DATA_ROOT)

print('\n특징 추출 중...')
X_feat = [extract_features(seq) for seq in X_raw]
print(f'특징 shape 예시: {X_feat[0].shape}  (프레임수 × 40차원)')

In [ ]:
# 셀 5: LOO 교차검증 (원본 5샘플로 현재 정확도 확인)
print('[LOO 교차검증 - 원본 데이터]')
loo = LeaveOneOut()
y_arr = np.array(y_raw)
correct = 0

for i, (tr_idx, te_idx) in enumerate(loo.split(X_feat)):
    clf = DTWKNNClassifier(k=5)
    clf.fit([X_feat[j] for j in tr_idx], y_arr[tr_idx].tolist())
    pred, conf, _ = clf.predict_one(X_feat[te_idx[0]])
    ok = pred == y_arr[te_idx[0]]
    if ok:
        correct += 1
    mark = '✓' if ok else '✗'
    print(f'  [{i+1:3d}] 정답={y_arr[te_idx[0]]:<6} 예측={pred:<6} {mark}')

print(f'\nLOO 정확도: {correct}/{len(X_feat)} = {correct/len(X_feat)*100:.1f}%')

In [ ]:
# 셀 6: 데이터 증강 (×15) + 최종 모델 학습
print('[데이터 증강 × 15]')
X_aug_raw, y_aug = augment_dataset(X_raw, y_raw, n=15)

print('특징 추출 중 (증강 데이터)...')
X_aug_feat = [extract_features(seq) for seq in X_aug_raw]

final_clf = DTWKNNClassifier(k=5)
final_clf.fit(X_aug_feat, y_aug)
print('최종 모델 학습 완료!')

In [ ]:
# 셀 7: 모델 저장 (Google Drive)
save_path = '/content/drive/MyDrive/ksl_dtw_knn_model.pkl'
joblib.dump({'classifier': final_clf, 'label_names': label_names}, save_path)
print(f'모델 저장 완료: {save_path}')

In [ ]:
# 셀 8: 테스트 - 첫 번째 샘플로 예측 확인
test_seq = X_raw[0]
test_feat = extract_features(test_seq)
pred, conf, top5 = final_clf.predict_one(test_feat)

print(f'실제 레이블: {y_raw[0]}')
print(f'예측 결과:   {pred}  (신뢰도: {conf*100:.1f}%)')
print('\nTop-5 유사도:')
for dist, lbl in top5:
    print(f'  {lbl}: {dist:.4f}')